In [1]:
import cv2
import numpy as np

In [2]:
hsv_data = {
  "blackNyellow_capsule": {
    "BLACK": {"lower": [0, 0, 0], "upper": [179, 255, 58], "open": 1, "close": 1},
    "YELLOW": {"lower": [16, 60, 81], "upper": [38, 251, 255], "open": 1, "close": 2}
  },
  "cream_capsule": {
    "Cream": {"lower": [63, 0, 82], "upper": [179, 26, 205], "open": 1, "close": 3},
    "BROWN": {"lower": [16, 28, 30], "upper": [20, 95, 210], "open": 1, "close": 2}
  },
  "green_capsule": {
    "GREEN": {"lower": [71, 25, 47], "upper": [88, 255, 255], "open": 1, "close": 1}
  },
  "orangeandblue": {
    "ORANGE": {"lower": [8, 80, 150], "upper": [24, 255, 255], "open": 1, "close": 1},
    "BLUE": {"lower": [85, 45, 50], "upper": [135, 255, 255], "open": 1, "close": 2}
  },
  "60429-203_M_LH3": {
    "GRAY": {"lower": [18, 0, 146], "upper": [145, 235, 255], "open": 1, "close": 6},
    "GREEN": {"lower": [35, 45, 60], "upper": [85, 255, 255], "open": 1, "close": 2}
  },
  "317220267": {
    "RED": {"lower": [3, 25, 49], "upper": [179, 255, 255], "open": 0, "close": 4}
  },
  "634810623": {
    "BLUE": {"lower": [85, 45, 50], "upper": [135, 255, 255], "open": 1, "close": 2}
  },
  "bloe_oval": {
    "BLUE": {"lower": [85, 80, 49], "upper": [144, 235, 141], "open": 1, "close": 2}
  },
  "brown_oval": {
    "BROWN": {"lower": [0, 13, 118], "upper": [179, 255, 255], "open": 0, "close": 2}
  },
  "green_oval": {
    "GREEN": {"lower": [0, 42, 61], "upper": [40, 126, 217], "open": 1, "close": 3}
  }
}

In [3]:
video = cv2.VideoCapture(1)

In [4]:
while True:
    success, img = video.read()
    if not success:
        break

    image_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Iteramos por cada objeto en tu JSON
    for item_name, colors in hsv_data.items():
        for color_name, range_data in colors.items():

            lower = np.array(range_data['lower'])
            upper = np.array(range_data['upper'])

            # Crear máscara para este color específico
            mask = cv2.inRange(image_hsv, lower, upper)

            # Aplicar operaciones morfológicas según tu JSON (open/close)
            if range_data.get('open', 0) > 0:
                kernel = np.ones((5,5), np.uint8)
                mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=range_data['open'])
            if range_data.get('close', 0) > 0:
                kernel = np.ones((5,5), np.uint8)
                mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=range_data['close'])

            # Encontrar contornos
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            for contour in contours:
                area = cv2.contourArea(contour)
                if area > 5000:
                    x, y, w, h = cv2.boundingRect(contour)

                    # Dibujar el rectángulo
                    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)

                    # Poner el nombre del objeto y color
                    label = f"{item_name}: {color_name}"
                    cv2.putText(img, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

    cv2.imshow('webcam', img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break